# 04 — Results Analysis & Visualization

Load benchmark results and generate comparison tables, charts, and key findings.

**Run this on**: Anywhere (no GPU needed)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

df = pd.read_csv("results/tables/all_results.csv")
df

## 1. Comparison Table (for README)

In [ ]:
# Format a clean comparison table
display_cols = ["model", "perplexity_wikitext", "perplexity_medical", "pubmedqa_accuracy", "tokens_per_sec", "peak_vram_gb"]
table = df[display_cols].copy()
table.columns = ["Model", "PPL (Wiki)", "PPL (Med)", "PubMedQA Acc", "Tok/s", "VRAM (GB)"]

# Calculate % change vs BF16 fine-tuned baseline
fp16_row = table[table["Model"] == "gemma4-e4b-medical-bf16"]
if len(fp16_row) > 0:
    fp16_ppl = fp16_row["PPL (Wiki)"].values[0]
    fp16_acc = fp16_row["PubMedQA Acc"].values[0]
    fp16_vram = fp16_row["VRAM (GB)"].values[0]
    
    table["PPL Change"] = ((table["PPL (Wiki)"] - fp16_ppl) / fp16_ppl * 100).round(1).astype(str) + "%"
    table["Acc Change"] = ((table["PubMedQA Acc"] - fp16_acc) / fp16_acc * 100).round(1).astype(str) + "%"
    table["VRAM Savings"] = ((fp16_vram - table["VRAM (GB)"]) / fp16_vram * 100).round(1).astype(str) + "%"

print(table.to_markdown(index=False))
table

## 2. Accuracy vs Memory Tradeoff

In [ ]:
import os
os.makedirs("results/figures", exist_ok=True)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

colors = {
    'bf16': '#333333',
    'gptq': '#e74c3c',
    'awq': '#3498db',
    'bnb': '#2ecc71',
}

for _, row in df.iterrows():
    name = row['model']
    if 'gptq' in name:
        c = colors['gptq']
    elif 'awq' in name:
        c = colors['awq']
    elif 'bnb' in name:
        c = colors['bnb']
    else:
        c = colors['bf16']
    
    ax.scatter(row['peak_vram_gb'], row['pubmedqa_accuracy'], s=120, c=c, zorder=5)
    label = name.replace('gemma4-e4b-', '').replace('med-', '')
    ax.annotate(label, 
                (row['peak_vram_gb'], row['pubmedqa_accuracy']),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel('Peak VRAM (GB)')
ax.set_ylabel('PubMedQA Accuracy')
ax.set_title('Accuracy vs Memory: Quantized Medical Gemma 4 E4B')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/accuracy_vs_memory.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Perplexity Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = df['model'].str.replace('gemma4-e4b-', '')

axes[0].barh(models, df['perplexity_wikitext'], color='steelblue')
axes[0].set_xlabel('Perplexity')
axes[0].set_title('WikiText-2 Perplexity (lower = better)')

axes[1].barh(models, df['perplexity_medical'], color='coral')
axes[1].set_xlabel('Perplexity')
axes[1].set_title('Medical Text Perplexity (lower = better)')

plt.tight_layout()
plt.savefig('results/figures/perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Speed Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(models, df['tokens_per_sec'], color='mediumseagreen')
ax.set_xlabel('Tokens per Second')
ax.set_title('Inference Throughput')
plt.tight_layout()
plt.savefig('results/figures/speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Findings Summary

In [ ]:
# Auto-generate key findings
quant_models = df[~df['model'].str.contains('bf16')]

best_acc = quant_models.loc[quant_models['pubmedqa_accuracy'].idxmax()]
best_speed = quant_models.loc[quant_models['tokens_per_sec'].idxmax()]
best_memory = quant_models.loc[quant_models['peak_vram_gb'].idxmin()]
best_ppl = quant_models.loc[quant_models['perplexity_wikitext'].idxmin()]

fp16_ft = df[df['model'] == 'gemma4-e4b-medical-bf16'].iloc[0] if 'gemma4-e4b-medical-bf16' in df['model'].values else None

print("KEY FINDINGS")
print("="*60)
print(f"Best accuracy retention: {best_acc['model']} ({best_acc['pubmedqa_accuracy']:.4f})")
print(f"Best throughput:         {best_speed['model']} ({best_speed['tokens_per_sec']:.1f} tok/s)")
print(f"Lowest memory:           {best_memory['model']} ({best_memory['peak_vram_gb']:.2f} GB)")
print(f"Best perplexity:         {best_ppl['model']} ({best_ppl['perplexity_wikitext']:.2f})")

if fp16_ft is not None:
    print(f"\nBF16 baseline VRAM:      {fp16_ft['peak_vram_gb']:.2f} GB")
    print(f"Best quantized VRAM:     {best_memory['peak_vram_gb']:.2f} GB")
    savings = (fp16_ft['peak_vram_gb'] - best_memory['peak_vram_gb']) / fp16_ft['peak_vram_gb'] * 100
    print(f"Memory reduction:        {savings:.1f}%")
    
    acc_drop = (fp16_ft['pubmedqa_accuracy'] - best_acc['pubmedqa_accuracy']) / fp16_ft['pubmedqa_accuracy'] * 100
    print(f"Accuracy drop (best):    {acc_drop:.1f}%")